In [1]:
import numpy as np
import pandas as pd
import warnings
import graphviz
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

In [2]:
class new_1_category_people():
    def __init__(self,df):
        self.df = pd.read_csv(df)


    def generate(self, N_new,target_column,target_value,categorical_cols,randomChoiceCategorialCols,bool_copy_categorial_col = True):

        df = self.df
        healthy = df[df[target_column] == target_value].copy()

        target_col = target_column
        numeric_cols = [c for c in healthy.columns 
                        if c not in categorical_cols + [target_col]]

        N_new = N_new
        new_people = []

        # Вычисляем стандартные отклонения для шума
        std_devs = healthy[numeric_cols].std()

        for _ in range(N_new):
            # Случайно выбираем исходного человека
            original = healthy.sample(n=1).iloc[0]
            
            # Добавляем шум (5% от стандартного отклонения)
            noise = np.random.normal(0, std_devs * 0.05, len(numeric_cols))
            
            person = {}
            for i, col in enumerate(numeric_cols):
                person[col] = max(0, original[col] + noise[i])  # не допускаем отрицательных
                person[col] = round(person[col], 2)
            
            # Категориальные признаки копируем с вероятностью 80% (небольшой шум)
            if categorical_cols and bool_copy_categorial_col:
                for i in categorical_cols:
                    if np.random.random() > 0.2:
                        person[i] = original[i]

                    else:
                        person[i] = np.random.choice(randomChoiceCategorialCols)
            else:
                pass
            
            person[target_column] = target_value
            new_people.append(person)
        
        self.df_new = pd.DataFrame(new_people)

        # self.df_new.to_csv('saved_data/synthetic_healthy_people.csv', index=False)
        self.df_new["name"] = range(96, 96 + len(self.df_new))

        print(f"✅ Сгенерировано {N_new} человек методом бутстрапа с шумом")
        print(self.df_new.head())
    def merge_new_df(self, merge_columns):
        df_new_1_category_people = pd.merge(
        self.df, 
        self.df_new, 
        how='outer', 
        on=merge_columns
        )
        df_new_1_category_people = df_new_1_category_people.drop("name",axis=1)
        return df_new_1_category_people

In [3]:
new_1_category_df_with_data_spliting = new_1_category_people("saved_data/mean_df_with_spliting.csv")
merge_columns = [
    # Целевая переменная
    "здоров(1)\болен(0)", 
    
    # Биохимия
    "ПОЛ в плазме", 
    "ПОЛ в мембр. Эритроцитов", 
    "Лактат", 
    "Глюкоза", 
    "Общий белок", 
    "Мочевина", 
    "Кортизол", 
    "Зонулин", 
    "Холестерин", 
    
    # Липидный профиль (уже без суффиксов _x, _y)
    "ЭХС", 
    "ТГ", 
    "НЭЖК", 
    "СХ", 
    "ФЛ", 
    "ЭХ",
    "НеЖК", 
    "ХЛ", 
    "СЖК",
    
    # Демография
    "Пол", 
    "Возраст"
]

new_1_category_df_with_data_spliting.generate(65,target_column = 'здоров(1)\болен(0)',target_value = 1,categorical_cols =  ['Пол'],randomChoiceCategorialCols = [1,2])
df_with_data_spliting_with_new_1_category = new_1_category_df_with_data_spliting.merge_new_df(merge_columns)
df_with_data_spliting_with_new_1_category
df_with_data_spliting_with_new_1_category.to_csv('saved_data/df_with_data_spliting_with_new_1_category.csv', index=False)

✅ Сгенерировано 65 человек методом бутстрапа с шумом
   ПОЛ в плазме  ПОЛ в мембр. Эритроцитов  Лактат  Глюкоза  Общий белок  \
0          1.59                      6.32   85.16     5.65         6.77   
1          1.22                      4.71   72.61     3.61         5.92   
2          2.03                      4.54   58.36     5.57         4.24   
3          2.02                      5.79   76.63     7.62         5.28   
4          1.98                      5.84   76.80     7.65         5.25   

   Мочевина  Кортизол  Зонулин  Холестерин    ЭХС  ...   НеЖК     ХЛ     ФЛ  \
0      1.58     20.47   193.45        0.28  12.97  ...  21.24  12.17  12.67   
1      3.04     28.20   293.48        0.27  17.52  ...  31.53  13.27  14.90   
2      1.53     25.24   238.67        0.63  13.72  ...  32.54  13.40  10.98   
3      2.65     30.90   228.52        0.21  21.74  ...  47.41  11.71  13.42   
4      2.58     31.40   227.84        0.20  21.72  ...  47.57  11.65  13.61   

     СЖК     ТГ     Э

In [4]:
new_1_category_df_WO_data_spliting = new_1_category_people("saved_data/mean_df_without_spliting_data.csv")
merge_columns = [
    # Целевая переменная
    "здоров(1)\болен(0)", 
    
    # Биохимия
    "ПОЛ в плазме", 
    "ПОЛ в мембр. Эритроцитов", 
    "Лактат", 
    "Глюкоза", 
    "Общий белок", 
    "Мочевина", 
    "Кортизол", 
    "Зонулин", 
    "Холестерин", 
    
    # Липидный профиль (уже без суффиксов _x, _y)
    "ЭХС", 
    "ТГ", 
    "НЭЖК", 
    "СХ", 
    "ФЛ", 
    "ЭХ",
    "НеЖК", 
    "ХЛ", 
    "СЖК",
    
    # Демография
    "Пол", 
    "Возраст"
]

new_1_category_df_WO_data_spliting.generate(65,target_column = 'здоров(1)\болен(0)',target_value = 1,categorical_cols =  ['Пол'],randomChoiceCategorialCols = [1,2])
df_WO_data_spliting_with_new_1_category = new_1_category_df_WO_data_spliting.merge_new_df(merge_columns)
df_WO_data_spliting_with_new_1_category.to_csv('saved_data/df_WO_data_spliting_with_new_1_category.csv', index=False)

✅ Сгенерировано 65 человек методом бутстрапа с шумом
   ПОЛ в плазме  ПОЛ в мембр. Эритроцитов  Лактат  Глюкоза  Общий белок  \
0          0.78                      4.57   76.51     6.70         3.89   
1          0.82                      5.27   75.05     4.11         4.20   
2          1.96                      5.75   76.64     7.70         5.33   
3          2.01                      5.27   75.08     5.48         3.15   
4          1.17                      5.36   77.58     5.02         4.11   

   Мочевина  Кортизол  Зонулин  Холестерин    ЭХС  ...   НеЖК     ХЛ     ФЛ  \
0      1.54     33.26   236.80        0.19  17.32  ...  46.68   8.87   9.25   
1      1.47     25.77   224.53        0.18  25.24  ...  23.59  11.23   7.22   
2      2.51     30.61   225.56        0.20  22.45  ...  47.03  11.59  12.57   
3      1.50     30.57   233.61        0.26  13.91  ...  23.37  20.25  17.73   
4      1.53     22.86   193.14        0.28  21.79  ...  27.84  11.47  13.34   

     СЖК     ТГ     С

In [5]:
dfdf = pd.read_csv("/home/ikuku/UDGU_project/main/EEG_files/data_for_EEG/combined_horizontal_data.csv")

new_1_category_EEG = new_1_category_people("/home/ikuku/UDGU_project/main/EEG_files/data_for_EEG/combined_horizontal_data.csv")

new_1_category_EEG.generate(98,target_column = 'здоров(1)/болен(0)',target_value = 1,categorical_cols =  ["Имя"],randomChoiceCategorialCols = [],bool_copy_categorial_col = False)
common_cols = list(set(new_1_category_EEG.df.columns) & set(new_1_category_EEG.df_new.columns))

df_with_new_1_category_EEG = new_1_category_EEG.merge_new_df(common_cols)
df_with_new_1_category_EEG = df_with_new_1_category_EEG.drop("Имя",axis=1)
df_with_new_1_category_EEG.to_csv('saved_data/df_with_new_1_category_EEG.csv', index=False)
df_with_new_1_category_EEG

✅ Сгенерировано 98 человек методом бутстрапа с шумом
   D_1:2  D_1:3  D_1:4  D_1:5  D_1:6  D_1:7  D_1:8  D_1:9  D_1:10  D_1:11  \
0   0.87   0.73   0.78   0.66   0.44   0.71   0.60   0.57    0.49    0.44   
1   0.90   0.85   0.73   0.81   0.76   0.74   0.59   0.64    0.64    0.51   
2   0.95   0.93   0.85   0.85   0.87   0.70   0.69   0.67    0.69    0.68   
3   0.83   0.84   0.74   0.87   0.91   0.71   0.64   0.59    0.70    0.64   
4   0.88   0.89   0.89   0.98   0.84   0.83   0.90   0.61    0.73    0.65   

   ...  B2_17:20  B2_17:21  B2_18:19  B2_18:20  B2_18:21  B2_19:20  B2_19:21  \
0  ...      0.53      0.48      0.39      0.55      0.61      0.63      0.51   
1  ...      0.79      0.76      0.64      0.82      0.93      0.63      0.65   
2  ...      0.81      0.76      0.51      0.57      0.64      0.73      0.60   
3  ...      0.66      0.61      0.59      0.70      0.65      0.63      0.58   
4  ...      0.83      0.83      0.69      0.82      0.81      0.74      0.73   

   

,D_1:2,D_1:3,D_1:4,D_1:5,D_1:6,D_1:7,D_1:8,D_1:9,D_1:10,D_1:11,...,B2_17:19,B2_17:20,B2_17:21,B2_18:19,B2_18:20,B2_18:21,B2_19:20,B2_19:21,B2_20:21,здоров(1)/болен(0)
0,0.875,0.875,0.758,0.892,0.888,0.855,0.719,0.638,0.548,0.639,...,0.565,0.908,0.948,0.640,0.742,0.791,0.655,0.631,0.978,0
1,0.870,0.730,0.780,0.660,0.440,0.710,0.600,0.570,0.490,0.440,...,0.430,0.530,0.480,0.390,0.550,0.610,0.630,0.510,0.910,1
2,0.601,0.632,0.861,0.909,0.924,0.602,0.557,0.824,0.858,0.519,...,0.504,0.929,0.883,0.556,0.751,0.699,0.525,0.498,0.923,0
3,0.860,0.740,0.790,0.650,0.420,0.710,0.600,0.570,0.490,0.440,...,0.420,0.530,0.490,0.380,0.540,0.600,0.630,0.520,0.910,1
4,0.870,0.730,0.790,0.660,0.420,0.710,0.610,0.570,0.490,0.450,...,0.420,0.520,0.490,0.380,0.540,0.600,0.620,0.520,0.910,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,0.940,0.930,0.850,0.850,0.860,0.690,0.710,0.670,0.690,0.680,...,0.670,0.810,0.740,0.510,0.580,0.640,0.730,0.600,0.830,1
280,0.932,0.926,0.852,0.794,0.782,0.795,0.809,0.692,0.628,0.526,...,0.547,0.877,0.874,0.558,0.878,0.882,0.583,0.575,0.999,0
281,0.923,0.909,0.864,0.847,0.775,0.794,0.851,0.746,0.711,0.707,...,0.555,0.943,0.958,0.545,0.935,0.970,0.583,0.573,0.974,0
282,0.834,0.720,0.726,0.792,0.736,0.724,0.746,0.688,0.739,0.680,...,0.612,0.791,0.616,0.561,0.627,0.556,0.601,0.542,0.585,0
